In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

RANDOM_STATE = 42

In [4]:
PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

modeling_data = pd.read_csv(
    DATA_INTERIM / "modeling_data.csv",
    dtype={"FIPS": "string"}
)

print("Dataset shape:", modeling_data.shape)
print("Unique FIPS:", modeling_data["FIPS"].nunique())
print("Missing target values:", modeling_data["OBESITY_AdjPrev"].isna().sum())

Dataset shape: (3135, 22)
Unique FIPS: 3135
Missing target values: 0


In [5]:
selected_predictors = [
    "PCT_LACCESS_POP19",
    "PCT_LACCESS_LOWI19",
    "GROCPTH20",
    "CONVSPTH20",
    "FFRPTH20",
    "FSRPTH20",
    "MEDHHINC21",
    "POVRATE21",
    "CHILDPOVRATE21",
    "DEEPPOVRATE21",
    "PC_SNAPBEN22",
    "PCT_65OLDER20",
    "PCT_18YOUNGER20",
    "PCT_NHWHITE20",
    "PCT_NHBLACK20",
    "PCT_HISP20",
    "PCT_NHASIAN20",
    "RECFACPTH20"
]

X = modeling_data[selected_predictors].copy()
y = modeling_data["OBESITY_AdjPrev"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Predictors:", len(selected_predictors))

X shape: (3135, 18)
y shape: (3135,)
Predictors: 18


In [6]:
outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Outer CV folds:", outer_cv.get_n_splits())
print("Inner CV folds:", inner_cv.get_n_splits())

Outer CV folds: 5
Inner CV folds: 3


In [7]:
class ThesisPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, missing_threshold=20.0, correlation_threshold=0.80):
        self.missing_threshold = missing_threshold
        self.correlation_threshold = correlation_threshold

    def fit(self, X, y=None):
        X = X.copy()

        # 1. Missingness assessment
        self.missing_summary_ = pd.DataFrame({
            "missing_count": X.isna().sum(),
            "missing_pct": X.isna().mean() * 100
        }).round(2)

        self.excluded_missing_ = self.missing_summary_[
            self.missing_summary_["missing_pct"] > self.missing_threshold
        ].index.tolist()

        self.retained_after_missing_ = [
            col for col in X.columns
            if col not in self.excluded_missing_
        ]

        X_retained = X[self.retained_after_missing_].copy()

        # 2. Median imputation
        self.imputer_ = SimpleImputer(strategy="median")
        self.imputer_.fit(X_retained)

        X_imputed = pd.DataFrame(
            self.imputer_.transform(X_retained),
            columns=self.retained_after_missing_,
            index=X.index
        )

        # 3. Pearson correlation screening
        self.corr_matrix_ = X_imputed.corr(method="pearson")

        high_corr_pairs = []

        columns = self.corr_matrix_.columns

        for i in range(len(columns)):
            for j in range(i + 1, len(columns)):
                r = self.corr_matrix_.iloc[i, j]

                if abs(r) >= self.correlation_threshold:
                    high_corr_pairs.append({
                        "predictor_1": columns[i],
                        "predictor_2": columns[j],
                        "r": r,
                        "abs_r": abs(r)
                    })

        self.high_corr_pairs_ = pd.DataFrame(high_corr_pairs)

        self.correlation_excluded_ = []

        if (
            "POVRATE21" in X_imputed.columns
            and "CHILDPOVRATE21" in X_imputed.columns
            and abs(
                self.corr_matrix_.loc[
                    "POVRATE21",
                    "CHILDPOVRATE21"
                ]
            ) >= self.correlation_threshold
        ):
            self.correlation_excluded_.append(
                "CHILDPOVRATE21"
            )

        if (
            "PCT_LACCESS_POP19" in X_imputed.columns
            and "PCT_LACCESS_LOWI19" in X_imputed.columns
            and abs(
                self.corr_matrix_.loc[
                    "PCT_LACCESS_POP19",
                    "PCT_LACCESS_LOWI19"
                ]
            ) >= self.correlation_threshold
        ):
            self.correlation_excluded_.append(
                "PCT_LACCESS_POP19"
            )

        self.retained_predictors_ = [
            col for col in X_imputed.columns
            if col not in self.correlation_excluded_
        ]

        X_final = X_imputed[self.retained_predictors_].copy()

        # 4. IQR review only
        iqr_summary = []

        for column in X_final.columns:
            values = X_final[column]

            q1 = values.quantile(0.25)
            q3 = values.quantile(0.75)
            iqr = q3 - q1

            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            outlier_mask = (
                (values < lower_bound)
                | (values > upper_bound)
            )

            iqr_summary.append({
                "predictor": column,
                "Q1": q1,
                "Q3": q3,
                "IQR": iqr,
                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "outlier_count": int(outlier_mask.sum()),
                "min": values.min(),
                "max": values.max()
            })

        self.iqr_summary_ = pd.DataFrame(iqr_summary)

        return self

    def transform(self, X):
        X = X.copy()

        X_retained = X[self.retained_after_missing_].copy()

        X_imputed = pd.DataFrame(
            self.imputer_.transform(X_retained),
            columns=self.retained_after_missing_,
            index=X.index
        )

        return X_imputed[self.retained_predictors_].copy()

In [8]:
lr_pipeline = Pipeline([
    (
        "preprocessor",
        ThesisPreprocessor(
            missing_threshold=20.0,
            correlation_threshold=0.80
        )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "lr",
        LinearRegression()
    )
])

rf_pipeline = Pipeline([
    (
        "preprocessor",
        ThesisPreprocessor(
            missing_threshold=20.0,
            correlation_threshold=0.80
        )
    ),
    (
        "rf",
        RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

xgb_pipeline = Pipeline([
    (
        "preprocessor",
        ThesisPreprocessor(
            missing_threshold=20.0,
            correlation_threshold=0.80
        )
    ),
    (
        "xgb",
        XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

print("Base-model pipelines created.")

Base-model pipelines created.


In [9]:
rf_param_distributions = {
    "rf__n_estimators": [100, 200, 300, 400, 500],
    "rf__max_depth": [None, 5, 10, 15, 20, 30],
    "rf__min_samples_split": [2, 5, 10]
}

xgb_param_distributions = {
    "xgb__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "xgb__n_estimators": [100, 200, 300, 400, 500],
    "xgb__max_depth": [2, 3, 4, 5, 6],
    "xgb__subsample": [0.6, 0.8, 1.0]
}

print("Random Forest search space:")
for parameter, values in rf_param_distributions.items():
    print(parameter, ":", values)

print("\nXGBoost search space:")
for parameter, values in xgb_param_distributions.items():
    print(parameter, ":", values)

Random Forest search space:
rf__n_estimators : [100, 200, 300, 400, 500]
rf__max_depth : [None, 5, 10, 15, 20, 30]
rf__min_samples_split : [2, 5, 10]

XGBoost search space:
xgb__learning_rate : [0.01, 0.05, 0.1, 0.2]
xgb__n_estimators : [100, 200, 300, 400, 500]
xgb__max_depth : [2, 3, 4, 5, 6]
xgb__subsample : [0.6, 0.8, 1.0]


In [10]:
train_idx, val_idx = next(outer_cv.split(X))

X_train_outer = X.iloc[train_idx].copy()
X_val_outer = X.iloc[val_idx].copy()

y_train_outer = y.iloc[train_idx].copy()
y_val_outer = y.iloc[val_idx].copy()

print("Outer Fold 1")
print("Outer training samples:", len(X_train_outer))
print("Outer validation samples:", len(X_val_outer))

Outer Fold 1
Outer training samples: 2508
Outer validation samples: 627


In [11]:
rf_search_outer = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_distributions,
    n_iter=10,
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True
)

xgb_search_outer = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_distributions,
    n_iter=10,
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True
)

print("Tuning Random Forest...")
rf_search_outer.fit(
    X_train_outer,
    y_train_outer
)

print("Tuning XGBoost...")
xgb_search_outer.fit(
    X_train_outer,
    y_train_outer
)

print("\nBest RF parameters:")
print(rf_search_outer.best_params_)

print("\nBest XGBoost parameters:")
print(xgb_search_outer.best_params_)

Tuning Random Forest...
Tuning XGBoost...

Best RF parameters:
{'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__max_depth': None}

Best XGBoost parameters:
{'xgb__subsample': 0.8, 'xgb__n_estimators': 200, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.05}


In [12]:
stack_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_lr = np.zeros(len(X_train_outer))
oof_rf = np.zeros(len(X_train_outer))
oof_xgb = np.zeros(len(X_train_outer))

print("Stacking OOF folds:", stack_cv.get_n_splits())
print("OOF prediction slots per model:", len(oof_lr))

for stack_fold, (stack_train_idx, stack_val_idx) in enumerate(
    stack_cv.split(X_train_outer),
    start=1
):
    print(
        f"Stack Fold {stack_fold}: "
        f"train={len(stack_train_idx)}, "
        f"validation={len(stack_val_idx)}"
    )

Stacking OOF folds: 3
OOF prediction slots per model: 2508
Stack Fold 1: train=1672, validation=836
Stack Fold 2: train=1672, validation=836
Stack Fold 3: train=1672, validation=836


In [13]:
from sklearn.base import clone

In [14]:
# Best hyperparameters selected from the complete outer training portion
best_rf_params = rf_search_outer.best_params_
best_xgb_params = xgb_search_outer.best_params_

# Reset OOF prediction arrays
oof_lr = np.zeros(len(X_train_outer))
oof_rf = np.zeros(len(X_train_outer))
oof_xgb = np.zeros(len(X_train_outer))

for stack_fold, (stack_train_idx, stack_val_idx) in enumerate(
    stack_cv.split(X_train_outer),
    start=1
):

    print(f"Generating OOF predictions for Stack Fold {stack_fold}...")

    # Split the OUTER TRAINING portion again
    X_stack_train = X_train_outer.iloc[stack_train_idx].copy()
    X_stack_val = X_train_outer.iloc[stack_val_idx].copy()

    y_stack_train = y_train_outer.iloc[stack_train_idx].copy()

    # -------------------------
    # Linear Regression
    # -------------------------
    lr_model = clone(lr_pipeline)

    lr_model.fit(
        X_stack_train,
        y_stack_train
    )

    oof_lr[stack_val_idx] = lr_model.predict(
        X_stack_val
    )

    # -------------------------
    # Random Forest
    # -------------------------
    rf_model = clone(rf_pipeline)
    rf_model.set_params(**best_rf_params)

    rf_model.fit(
        X_stack_train,
        y_stack_train
    )

    oof_rf[stack_val_idx] = rf_model.predict(
        X_stack_val
    )

    # -------------------------
    # XGBoost
    # -------------------------
    xgb_model = clone(xgb_pipeline)
    xgb_model.set_params(**best_xgb_params)

    xgb_model.fit(
        X_stack_train,
        y_stack_train
    )

    oof_xgb[stack_val_idx] = xgb_model.predict(
        X_stack_val
    )

print("\nOOF generation complete.")
print("LR predictions:", len(oof_lr))
print("RF predictions:", len(oof_rf))
print("XGB predictions:", len(oof_xgb))

Generating OOF predictions for Stack Fold 1...
Generating OOF predictions for Stack Fold 2...
Generating OOF predictions for Stack Fold 3...

OOF generation complete.
LR predictions: 2508
RF predictions: 2508
XGB predictions: 2508


In [17]:
meta_train = pd.DataFrame({
    "LR_pred": oof_lr,
    "RF_pred": oof_rf,
    "XGB_pred": oof_xgb
})

print("Meta-training shape:", meta_train.shape)

print("\nMissing values:")
print(meta_train.isna().sum())

print("\nZero values:")
print((meta_train == 0).sum())

print("\nPreview:")
display(meta_train.head())

Meta-training shape: (2508, 3)

Missing values:
LR_pred     0
RF_pred     0
XGB_pred    0
dtype: int64

Zero values:
LR_pred     0
RF_pred     0
XGB_pred    0
dtype: int64

Preview:


,LR_pred,RF_pred,XGB_pred
0,36.178506,35.553,36.968624
1,43.546774,44.113,44.034428
2,41.710838,41.154,41.447666
3,39.323362,38.801,39.075375
4,46.015109,47.221,47.007404


In [18]:
meta_learner = LinearRegression()

meta_learner.fit(
    meta_train,
    y_train_outer
)

print("Meta-learner trained.")

print("\nMeta-learner coefficients:")
print("LR:", meta_learner.coef_[0])
print("RF:", meta_learner.coef_[1])
print("XGB:", meta_learner.coef_[2])

print("\nIntercept:")
print(meta_learner.intercept_)

Meta-learner trained.

Meta-learner coefficients:
LR: 0.1273802679322222
RF: 0.2185135967159861
XGB: 0.7009490164661022

Intercept:
-1.7894621954421908


In [19]:
final_lr = clone(lr_pipeline)

final_rf = clone(rf_pipeline)
final_rf.set_params(**best_rf_params)

final_xgb = clone(xgb_pipeline)
final_xgb.set_params(**best_xgb_params)

print("Fitting final LR...")
final_lr.fit(
    X_train_outer,
    y_train_outer
)

print("Fitting final RF...")
final_rf.fit(
    X_train_outer,
    y_train_outer
)

print("Fitting final XGBoost...")
final_xgb.fit(
    X_train_outer,
    y_train_outer
)

print("\nFinal base models fitted on full outer training data.")

Fitting final LR...
Fitting final RF...
Fitting final XGBoost...

Final base models fitted on full outer training data.


In [20]:
outer_lr_pred = final_lr.predict(X_val_outer)
outer_rf_pred = final_rf.predict(X_val_outer)
outer_xgb_pred = final_xgb.predict(X_val_outer)

meta_val = pd.DataFrame({
    "LR_pred": outer_lr_pred,
    "RF_pred": outer_rf_pred,
    "XGB_pred": outer_xgb_pred
})

print("Meta-validation shape:", meta_val.shape)
print("\nMissing values:")
print(meta_val.isna().sum())

print("\nPreview:")
display(meta_val.head())

Meta-validation shape: (627, 3)

Missing values:
LR_pred     0
RF_pred     0
XGB_pred    0
dtype: int64

Preview:


,LR_pred,RF_pred,XGB_pred
0,38.542913,38.567,38.036541
1,40.021657,39.554,40.147243
2,44.435121,44.765,44.798050
3,40.705231,39.669,39.923801
4,38.834156,37.342,38.539146


In [21]:
stack_pred = meta_learner.predict(meta_val)

print("Stacking predictions:", len(stack_pred))
print("Missing predictions:", np.isnan(stack_pred).sum())

print("\nFirst 5 stacking predictions:")
print(stack_pred[:5])

Stacking predictions: 627
Missing predictions: 0

First 5 stacking predictions:
[38.2092343  40.09276485 45.05360561 40.04834629 38.3309545 ]


In [22]:
stack_mae = mean_absolute_error(
    y_val_outer,
    stack_pred
)

stack_rmse = np.sqrt(
    mean_squared_error(
        y_val_outer,
        stack_pred
    )
)

stack_r2 = r2_score(
    y_val_outer,
    stack_pred
)

print("Stacking Ensemble - Outer Fold 1")
print(f"MAE:  {stack_mae:.4f}")
print(f"RMSE: {stack_rmse:.4f}")
print(f"R²:   {stack_r2:.4f}")

Stacking Ensemble - Outer Fold 1
MAE:  2.2820
RMSE: 2.9166
R²:   0.6177


In [23]:
stacking_fold_results = []
stacking_predictions = []
stacking_meta_results = []
stacking_hyperparameters = []

print("Storage initialized for 5-fold stacking evaluation.")

Storage initialized for 5-fold stacking evaluation.


In [24]:
for outer_fold, (train_idx, val_idx) in enumerate(
    outer_cv.split(X),
    start=1
):
    print(f"\n{'=' * 60}")
    print(f"OUTER FOLD {outer_fold}")
    print(f"{'=' * 60}")

    # -------------------------------------------------
    # 1. Outer train / validation split
    # -------------------------------------------------
    X_train_outer = X.iloc[train_idx].copy()
    X_val_outer = X.iloc[val_idx].copy()

    y_train_outer = y.iloc[train_idx].copy()
    y_val_outer = y.iloc[val_idx].copy()

    print(
        f"Outer training: {len(X_train_outer)} | "
        f"Outer validation: {len(X_val_outer)}"
    )

    # -------------------------------------------------
    # 2. Tune Random Forest
    # -------------------------------------------------
    rf_search = RandomizedSearchCV(
        estimator=clone(rf_pipeline),
        param_distributions=rf_param_distributions,
        n_iter=10,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True
    )

    print("Tuning Random Forest...")
    rf_search.fit(
        X_train_outer,
        y_train_outer
    )

    best_rf_params = rf_search.best_params_

    # -------------------------------------------------
    # 3. Tune XGBoost
    # -------------------------------------------------
    xgb_search = RandomizedSearchCV(
        estimator=clone(xgb_pipeline),
        param_distributions=xgb_param_distributions,
        n_iter=10,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True
    )

    print("Tuning XGBoost...")
    xgb_search.fit(
        X_train_outer,
        y_train_outer
    )

    best_xgb_params = xgb_search.best_params_

    print("Best RF:", best_rf_params)
    print("Best XGB:", best_xgb_params)

    # Save selected hyperparameters
    stacking_hyperparameters.append({
        "Outer_Fold": outer_fold,
        "RF_n_estimators": best_rf_params["rf__n_estimators"],
        "RF_max_depth": best_rf_params["rf__max_depth"],
        "RF_min_samples_split": best_rf_params["rf__min_samples_split"],
        "XGB_learning_rate": best_xgb_params["xgb__learning_rate"],
        "XGB_n_estimators": best_xgb_params["xgb__n_estimators"],
        "XGB_max_depth": best_xgb_params["xgb__max_depth"],
        "XGB_subsample": best_xgb_params["xgb__subsample"]
    })

    # -------------------------------------------------
    # 4. Generate OOF predictions for meta-learner
    # -------------------------------------------------
    oof_lr = np.zeros(len(X_train_outer))
    oof_rf = np.zeros(len(X_train_outer))
    oof_xgb = np.zeros(len(X_train_outer))

    for stack_fold, (stack_train_idx, stack_val_idx) in enumerate(
        stack_cv.split(X_train_outer),
        start=1
    ):
        X_stack_train = X_train_outer.iloc[stack_train_idx].copy()
        X_stack_val = X_train_outer.iloc[stack_val_idx].copy()

        y_stack_train = y_train_outer.iloc[stack_train_idx].copy()

        # Linear Regression
        lr_model = clone(lr_pipeline)

        lr_model.fit(
            X_stack_train,
            y_stack_train
        )

        oof_lr[stack_val_idx] = lr_model.predict(
            X_stack_val
        )

        # Random Forest
        rf_model = clone(rf_pipeline)
        rf_model.set_params(**best_rf_params)

        rf_model.fit(
            X_stack_train,
            y_stack_train
        )

        oof_rf[stack_val_idx] = rf_model.predict(
            X_stack_val
        )

        # XGBoost
        xgb_model = clone(xgb_pipeline)
        xgb_model.set_params(**best_xgb_params)

        xgb_model.fit(
            X_stack_train,
            y_stack_train
        )

        oof_xgb[stack_val_idx] = xgb_model.predict(
            X_stack_val
        )

    # -------------------------------------------------
    # 5. Train Linear Regression meta-learner
    # -------------------------------------------------
    meta_train = pd.DataFrame({
        "LR_pred": oof_lr,
        "RF_pred": oof_rf,
        "XGB_pred": oof_xgb
    })

    meta_learner = LinearRegression()

    meta_learner.fit(
        meta_train,
        y_train_outer
    )

    stacking_meta_results.append({
        "Outer_Fold": outer_fold,
        "LR_Coefficient": meta_learner.coef_[0],
        "RF_Coefficient": meta_learner.coef_[1],
        "XGB_Coefficient": meta_learner.coef_[2],
        "Intercept": meta_learner.intercept_
    })

    # -------------------------------------------------
    # 6. Refit base models on complete outer training
    # -------------------------------------------------
    final_lr = clone(lr_pipeline)

    final_rf = clone(rf_pipeline)
    final_rf.set_params(**best_rf_params)

    final_xgb = clone(xgb_pipeline)
    final_xgb.set_params(**best_xgb_params)

    final_lr.fit(
        X_train_outer,
        y_train_outer
    )

    final_rf.fit(
        X_train_outer,
        y_train_outer
    )

    final_xgb.fit(
        X_train_outer,
        y_train_outer
    )

    # -------------------------------------------------
    # 7. Predict untouched outer validation fold
    # -------------------------------------------------
    outer_lr_pred = final_lr.predict(X_val_outer)
    outer_rf_pred = final_rf.predict(X_val_outer)
    outer_xgb_pred = final_xgb.predict(X_val_outer)

    meta_val = pd.DataFrame({
        "LR_pred": outer_lr_pred,
        "RF_pred": outer_rf_pred,
        "XGB_pred": outer_xgb_pred
    })

    stack_pred = meta_learner.predict(meta_val)

    # -------------------------------------------------
    # 8. Evaluate stacking ensemble
    # -------------------------------------------------
    stack_mae = mean_absolute_error(
        y_val_outer,
        stack_pred
    )

    stack_rmse = np.sqrt(
        mean_squared_error(
            y_val_outer,
            stack_pred
        )
    )

    stack_r2 = r2_score(
        y_val_outer,
        stack_pred
    )

    stacking_fold_results.append({
        "Outer_Fold": outer_fold,
        "MAE": stack_mae,
        "RMSE": stack_rmse,
        "R2": stack_r2
    })

    # -------------------------------------------------
    # 9. Save county-level outer predictions
    # -------------------------------------------------
    fold_predictions = pd.DataFrame({
        "FIPS": modeling_data.iloc[val_idx]["FIPS"].values,
        "Actual": y_val_outer.values,
        "LR_Prediction": outer_lr_pred,
        "RF_Prediction": outer_rf_pred,
        "XGB_Prediction": outer_xgb_pred,
        "Stacking_Prediction": stack_pred,
        "Outer_Fold": outer_fold
    })

    stacking_predictions.append(
        fold_predictions
    )

    print(
        f"Fold {outer_fold} | "
        f"MAE={stack_mae:.4f} | "
        f"RMSE={stack_rmse:.4f} | "
        f"R²={stack_r2:.4f}"
    )

print("\nAll 5 outer folds completed.")


OUTER FOLD 1
Outer training: 2508 | Outer validation: 627
Tuning Random Forest...
Tuning XGBoost...
Best RF: {'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__max_depth': None}
Best XGB: {'xgb__subsample': 0.8, 'xgb__n_estimators': 200, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.05}
Fold 1 | MAE=2.2820 | RMSE=2.9166 | R²=0.6177

OUTER FOLD 2
Outer training: 2508 | Outer validation: 627
Tuning Random Forest...
Tuning XGBoost...
Best RF: {'rf__n_estimators': 100, 'rf__min_samples_split': 5, 'rf__max_depth': 20}
Best XGB: {'xgb__subsample': 0.8, 'xgb__n_estimators': 200, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.05}
Fold 2 | MAE=2.2334 | RMSE=2.8603 | R²=0.6498

OUTER FOLD 3
Outer training: 2508 | Outer validation: 627
Tuning Random Forest...


/Users/chelsearose/Documents/GitHub/county-obesity-prediction/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/chelsearose/Documents/GitHub/county-obesity-prediction/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/chelsearose/Documents/GitHub/county-obesity-prediction/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration 

Tuning XGBoost...
Best RF: {'rf__n_estimators': 100, 'rf__min_samples_split': 5, 'rf__max_depth': 20}
Best XGB: {'xgb__subsample': 0.8, 'xgb__n_estimators': 200, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.05}
Fold 3 | MAE=2.2202 | RMSE=2.9209 | R²=0.6112

OUTER FOLD 4
Outer training: 2508 | Outer validation: 627
Tuning Random Forest...
Tuning XGBoost...
Best RF: {'rf__n_estimators': 100, 'rf__min_samples_split': 5, 'rf__max_depth': 20}
Best XGB: {'xgb__subsample': 0.8, 'xgb__n_estimators': 200, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.05}
Fold 4 | MAE=2.0621 | RMSE=2.5856 | R²=0.6763

OUTER FOLD 5
Outer training: 2508 | Outer validation: 627
Tuning Random Forest...
Tuning XGBoost...
Best RF: {'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__max_depth': None}
Best XGB: {'xgb__subsample': 0.8, 'xgb__n_estimators': 200, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.05}
Fold 5 | MAE=2.1109 | RMSE=2.7300 | R²=0.5995

All 5 outer folds completed.


In [25]:
stacking_results_df = pd.DataFrame(stacking_fold_results)

stacking_summary = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Mean": [
        stacking_results_df["MAE"].mean(),
        stacking_results_df["RMSE"].mean(),
        stacking_results_df["R2"].mean()
    ],
    "SD": [
        stacking_results_df["MAE"].std(),
        stacking_results_df["RMSE"].std(),
        stacking_results_df["R2"].std()
    ]
})

print("Stacking Ensemble Performance")
print(stacking_summary.round(4))

Stacking Ensemble Performance
  Metric    Mean      SD
0    MAE  2.1817  0.0916
1   RMSE  2.8027  0.1438
2     R2  0.6309  0.0315


In [27]:
stacking_predictions_df = pd.concat(
    stacking_predictions,
    ignore_index=True
)

print("Total predictions:", len(stacking_predictions_df))
print("Unique FIPS:", stacking_predictions_df["FIPS"].nunique())
print("Duplicate FIPS:", stacking_predictions_df["FIPS"].duplicated().sum())
print(
    "Missing predictions:",
    stacking_predictions_df["Stacking_Prediction"].isna().sum()
)

print("\nPredictions per outer fold:")
print(
    stacking_predictions_df["Outer_Fold"]
    .value_counts()
    .sort_index()
)

display(stacking_predictions_df.head())

Total predictions: 3135
Unique FIPS: 3135
Duplicate FIPS: 0
Missing predictions: 0

Predictions per outer fold:
Outer_Fold
1    627
2    627
3    627
4    627
5    627
Name: count, dtype: int64


,FIPS,Actual,LR_Prediction,RF_Prediction,XGB_Prediction,Stacking_Prediction,Outer_Fold
0,01001,38.4,38.542913,38.567,38.036541,38.209234,1
1,01029,39.4,40.021657,39.554,40.147243,40.092765,1
2,01035,44.7,44.435121,44.765,44.798050,45.053606,1
3,01045,44.3,40.705231,39.669,39.923801,40.048346,1
4,01051,36.4,38.834156,37.342,38.539146,38.330954,1


In [28]:
# Create output directory
STACKING_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "stacking"
STACKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Convert stored results to DataFrames
stacking_hyperparameters_df = pd.DataFrame(stacking_hyperparameters)
stacking_meta_results_df = pd.DataFrame(stacking_meta_results)

# Save outer-fold metrics
stacking_results_df.to_csv(
    STACKING_OUTPUT_DIR / "stacking_outer_fold_metrics.csv",
    index=False
)

# Save all county-level predictions
stacking_predictions_df.to_csv(
    STACKING_OUTPUT_DIR / "stacking_outer_predictions.csv",
    index=False
)

# Save RF/XGB hyperparameters selected in each outer fold
stacking_hyperparameters_df.to_csv(
    STACKING_OUTPUT_DIR / "stacking_best_hyperparameters.csv",
    index=False
)

# Save meta-learner coefficients
stacking_meta_results_df.to_csv(
    STACKING_OUTPUT_DIR / "stacking_meta_coefficients.csv",
    index=False
)

# Save mean ± SD performance
stacking_summary.to_csv(
    STACKING_OUTPUT_DIR / "stacking_performance_summary.csv",
    index=False
)

print("Stacking outputs saved:")
for file in sorted(STACKING_OUTPUT_DIR.iterdir()):
    print("-", file.name)

Stacking outputs saved:
- stacking_best_hyperparameters.csv
- stacking_meta_coefficients.csv
- stacking_outer_fold_metrics.csv
- stacking_outer_predictions.csv
- stacking_performance_summary.csv
